In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib
from sklearn.metrics import accuracy_score,precision_score,precision_recall_curve,f1_score,roc_auc_score,confusion_matrix

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'Ecommerce_Dataset.csv' in files:
        file_path = os.path.join(root, 'Ecommerce_Dataset.csv')
        print("Dataset found at:", file_path)
        break




Dataset found at: /content/drive/MyDrive/Ecommerce_Dataset.csv


In [24]:

data = pd.read_csv(file_path)
print(data.shape)
data.head()

(8366491, 6)


,user_id,product_id,event_type,brand,price,category_code
0,530394012,2701390,view,lg,1249.14,appliances.kitchen.refrigerators
1,563175289,1005122,view,apple,1027.59,electronics.smartphone
2,527560632,1003879,view,huawei,540.27,electronics.smartphone
3,556740096,4804660,view,xiaomi,23.05,electronics.audio.headphone
4,551479057,4804581,view,xiaomi,35.75,electronics.audio.headphone


In [ ]:
df = data.copy()
df = df.dropna(subset=["user_id", "product_id"])
df = df.drop_duplicates(
    subset=["user_id", "product_id", "event_type"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Unique users:", df["user_id"].nunique())
print("Unique products:", df["product_id"].nunique())
print("\nEvent counts:")
print(df["event_type"].value_counts())

Dataset shape: (7815281, 6)
Unique users: 2359630
Unique products: 66418

Event counts:
event_type
view        7613348
purchase     201933
Name: count, dtype: int64


In [26]:
print(df[["user_id", "product_id", "event_type"]].isna().sum())


user_id       0
product_id    0
event_type    0
dtype: int64


In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)

# Using the 20,000 most active users
top_users = df["user_id"].value_counts().head(20_000).index
work = df[df["user_id"].isin(top_users)].copy()

# Treat purchases as stronger positive signals than views
work["event_weight"] = work["event_type"].map({
    "view": 1.0,
    "purchase": 3.0
}).fillna(1.0)

# Keep one row per user-product pair, retaining the strongest event
interactions = (
    work.groupby(["user_id", "product_id"], as_index=False)
        .agg(interaction_weight=("event_weight", "max"))
)

# Keep users with at least 2 distinct products
user_counts = interactions.groupby("user_id")["product_id"].nunique()
eligible_users = user_counts[user_counts >= 2].index
interactions = interactions[
    interactions["user_id"].isin(eligible_users)
].reset_index(drop=True)

print("Interactions:", len(interactions))
print("Users:", interactions["user_id"].nunique())
print("Products:", interactions["product_id"].nunique())
print(interactions["interaction_weight"].value_counts())

Interactions: 713747
Users: 20000
Products: 49515
interaction_weight
1.0    703339
3.0     10408
Name: count, dtype: int64


In [ ]:
train_rows, val_rows, test_rows = [], [], []

for _, group in interactions.groupby("user_id", sort=False):
    indices = group.index.to_numpy().copy()
    rng.shuffle(indices)

    n = len(indices)

    if n >= 3:
        # At least one training interaction; one validation and one test
        test_rows.append(indices[0])
        val_rows.append(indices[1])
        train_rows.extend(indices[2:])
    else:
        # Two interactions: one training, one test
        test_rows.append(indices[0])
        train_rows.extend(indices[1:])

train_pos = interactions.loc[train_rows].copy()
val_pos = interactions.loc[val_rows].copy()
test_pos = interactions.loc[test_rows].copy()

user_encoder = LabelEncoder()
product_encoder = LabelEncoder()

user_encoder.fit(train_pos["user_id"])
product_encoder.fit(train_pos["product_id"])

# Exclude held-out pairs whose products are absent from the training catalogue
def prepare_holdout(frame):
    frame = frame[
        frame["user_id"].isin(user_encoder.classes_)
        & frame["product_id"].isin(product_encoder.classes_)
    ].copy()

    frame["user_idx"] = user_encoder.transform(frame["user_id"])
    frame["product_idx"] = product_encoder.transform(frame["product_id"])
    return frame.reset_index(drop=True)

train_pos["user_idx"] = user_encoder.transform(train_pos["user_id"])
train_pos["product_idx"] = product_encoder.transform(train_pos["product_id"])

val_pos = prepare_holdout(val_pos)
test_pos = prepare_holdout(test_pos)

n_users = len(user_encoder.classes_)
n_items = len(product_encoder.classes_)

# All observed items per original user, to avoid false-negative sampling
seen = work.groupby("user_id")["product_id"].agg(set).to_dict()

print("Training positives:", len(train_pos))
print("Validation positives:", len(val_pos))
print("Test positives:", len(test_pos))
print("Training users:", n_users)
print("Training-catalogue products:", n_items)

Training positives: 673747
Validation positives: 19664
Test positives: 19710
Training users: 20000
Training-catalogue products: 48903


In [ ]:
def make_examples(pos_df, seed=42, weighted=False):
    local_rng = np.random.default_rng(seed)

    users = pos_df["user_idx"].to_numpy(dtype=np.int32)
    products = pos_df["product_idx"].to_numpy(dtype=np.int32)

    positive_labels = np.ones(len(pos_df), dtype=np.float32)
    negative_labels = np.zeros(len(pos_df), dtype=np.float32)

    # Give purchases greater influence in the training loss
    positive_weights = (
        pos_df["interaction_weight"].to_numpy(dtype=np.float32)
        if weighted else np.ones(len(pos_df), dtype=np.float32)
    )
    negative_weights = np.ones(len(pos_df), dtype=np.float32)

    negative_users, negative_products = [], []

    for u in users:
        original_user = user_encoder.classes_[u]
        observed = seen.get(original_user, set())

        while True:
            candidate = int(local_rng.integers(n_items))
            original_product = product_encoder.classes_[candidate]

            if original_product not in observed:
                break

        negative_users.append(u)
        negative_products.append(candidate)

    all_users = np.concatenate([
        users, np.asarray(negative_users, dtype=np.int32)
    ])
    all_products = np.concatenate([
        products, np.asarray(negative_products, dtype=np.int32)
    ])
    labels = np.concatenate([positive_labels, negative_labels])
    sample_weights = np.concatenate([positive_weights, negative_weights])

    order = local_rng.permutation(len(labels))

    return (
        all_users[order],
        all_products[order],
        labels[order],
        sample_weights[order]
    )

train_u, train_p, train_y, train_w = make_examples(
    train_pos, seed=SEED, weighted=True
)

val_u, val_p, val_y, val_w = make_examples(
    val_pos, seed=SEED + 1
)

test_u, test_p, test_y, test_w = make_examples(
    test_pos, seed=SEED + 2
)

print("Train examples:", len(train_y))
print("Validation examples:", len(val_y))
print("Test examples:", len(test_y))

Train examples: 1347494
Validation examples: 39328
Test examples: 39420


In [ ]:

embedding_dim = 32

user_input = layers.Input(
    shape=(1,), dtype="int32", name="user_id"
)
item_input = layers.Input(
    shape=(1,), dtype="int32", name="product_id"
)

user_vec = layers.Flatten()(
    layers.Embedding(n_users, embedding_dim)(user_input)
)
item_vec = layers.Flatten()(
    layers.Embedding(n_items, embedding_dim)(item_input)
)

x = layers.Concatenate()([user_vec, item_vec])
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(32, activation="relu")(x)
output = layers.Dense(1, activation="sigmoid")(x)

model = Model([user_input, item_input], output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc")]
)

history = model.fit(
    [train_u, train_p],
    train_y,
    sample_weight=train_w,
    validation_data=([val_u, val_p], val_y, val_w),
    epochs=5,
    batch_size=2048,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc",
            patience=2,
            mode="max",
            restore_best_weights=True
        )
    ],
    verbose=1
)

Epoch 1/5
658/658 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - auc: 0.8418 - loss: 0.4899 - val_auc: 0.8911 - val_loss: 0.4140
Epoch 2/5
658/658 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.8898 - loss: 0.4192 - val_auc: 0.9009 - val_loss: 0.3963
Epoch 3/5
658/658 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - auc: 0.9063 - loss: 0.3887 - val_auc: 0.9061 - val_loss: 0.3870
Epoch 4/5
658/658 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.9219 - loss: 0.3564 - val_auc: 0.9081 - val_loss: 0.3860
Epoch 5/5
658/658 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.9377 - loss: 0.3191 - val_auc: 0.9076 - val_loss: 0.3960


In [ ]:
K = 10
N_NEGATIVES = 100
rng = np.random.default_rng(42)

# Popularity baseline: count training interactions per product
popularity = (
    train_pos["product_idx"].value_counts().reindex(range(n_items), fill_value=0).to_numpy())

MAX_TEST_USERS = 2000
eval_pos = test_pos.sample(
    n=min(MAX_TEST_USERS, len(test_pos)),
    random_state=42
).reset_index(drop=True)

candidate_rows = []
user_rows = []

for row in eval_pos.itertuples(index=False):
    u = int(row.user_idx)
    target = int(row.product_idx)

    original_user = user_encoder.classes_[u]
    observed = seen.get(original_user, set())

    negatives = set()
    attempts = 0

    while len(negatives) < N_NEGATIVES and attempts < 5000:
        p = int(rng.integers(n_items))
        attempts += 1

        if (
            p != target
            and product_encoder.classes_[p] not in observed
        ):
            negatives.add(p)

    if len(negatives) < N_NEGATIVES:
        continue

    candidates = np.array(
        [target] + list(negatives),
        dtype=np.int32
    )

    candidate_rows.append(candidates)
    user_rows.append(u)

if not candidate_rows:
    raise ValueError("No eligible test cases. Check the interaction data.")

candidate_matrix = np.stack(candidate_rows)
user_matrix = np.repeat(
    np.array(user_rows, dtype=np.int32)[:, None],
    candidate_matrix.shape[1],
    axis=1
)

# Predict all user-product candidates in batches
scores = model.predict(
    [user_matrix.ravel(), candidate_matrix.ravel()],
    batch_size=8192,
    verbose=1
).reshape(candidate_matrix.shape)

# Popularity scores for the same candidates
pop_scores = popularity[candidate_matrix]


def calculate_metrics(score_matrix, k=10):
    recalls = []
    ndcgs = []

    for row_scores in score_matrix:
        # Relevant product is always at position 0
        rank = np.where(np.argsort(-row_scores, kind="stable") == 0)[0][0] + 1

        recalls.append(float(rank <= k))
        ndcgs.append(1 / np.log2(rank + 1) if rank <= k else 0.0)

    return np.mean(recalls), np.mean(ndcgs)


ncf_recall, ncf_ndcg = calculate_metrics(scores, K)
pop_recall, pop_ndcg = calculate_metrics(pop_scores, K)

print("Evaluated test cases:", len(candidate_rows))
print(f"NCF Recall@{K}:         {ncf_recall:.4f}")
print(f"NCF NDCG@{K}:           {ncf_ndcg:.4f}")
print(f"Popularity Recall@{K}:  {pop_recall:.4f}")
print(f"Popularity NDCG@{K}:    {pop_ndcg:.4f}")

25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
Evaluated test cases: 2000
NCF Recall@10:         0.7185
NCF NDCG@10:           0.4986
Popularity Recall@10:  0.6830
Popularity NDCG@10:    0.4691


In [ ]:
def recommend_products(user_id, top_k=10):

    # Check whether this user exists in our trained model
    user_classes = user_encoder.classes_
    user_match = np.where(user_classes == user_id)[0]

    if len(user_match) == 0:
        print("User not found in the trained model.")
        return pd.DataFrame()

    user_idx = int(user_match[0])

    # Products this user has already interacted with
    already_seen = seen.get(user_id, set())

    # Consider products from the training catalogue
    candidate_products = np.arange(n_items, dtype=np.int32)

    candidate_ids = product_encoder.inverse_transform(candidate_products)

    # Recommend only products the user has not interacted with
    mask = np.array([
        product_id not in already_seen
        for product_id in candidate_ids
    ])

    candidate_products = candidate_products[mask]

    if len(candidate_products) == 0:
        print("No unseen products available.")
        return pd.DataFrame()

    user_array = np.full(
        len(candidate_products), user_idx, dtype=np.int32
    )

    # Score all unseen products in batches
    scores = model.predict(
        [user_array, candidate_products],
        batch_size=8192,
        verbose=0
    ).ravel()

    top_indices = np.argsort(-scores)[:top_k]

    recommended_indices = candidate_products[top_indices]
    recommended_ids = product_encoder.inverse_transform(recommended_indices)

    return pd.DataFrame({
        "product_id": recommended_ids,
        "NCF_score": scores[top_indices]
    })


# Try 
sample_user = int(user_encoder.classes_[0])

recommendations = recommend_products(sample_user, top_k=10)
display(recommendations)

,product_id,NCF_score
0,1004767,0.999726
1,1004856,0.999554
2,1005160,0.999536
3,1004781,0.999481
4,1005159,0.999471
5,1002544,0.999418
6,1003306,0.999410
7,1004659,0.999375
8,1005115,0.999340
9,1005161,0.999334


In [ ]:
os.makedirs("artifacts", exist_ok=True)
# Save the trained TensorFlow model
model.save("artifacts/ncf_model.keras")
np.savez_compressed("ncf_weights.npz",*model.get_weights())

# Save encoders
joblib.dump(user_encoder, "artifacts/user_encoder.pkl")
joblib.dump(product_encoder, "artifacts/product_encoder.pkl")

# Save product metadata
product_details = (
    df[["product_id", "brand", "price", "category_code"]]
    .drop_duplicates("product_id")
)

product_details.to_pickle("artifacts/product_details.pkl")
joblib.dump(seen, "artifacts/seen_products.pkl")

print("Model and encoders saved successfully!")

Model and encoders saved successfully!
['ncf_model.keras', 'product_details.pkl', 'user_encoder.pkl', 'product_encoder.pkl', 'seen_products.pkl']
